# Business Problem
In the ride-hailing and taxi industry, fleet managers and independent drivers often optimize for "total fare volume" rather than "financial yield." Drivers waste fuel and time (deadheading) by cruising in areas that generate trips with low revenue-per-mile. Because standard meters have fixed base drops and time-based charges, a $15 trip that covers 1 mile in dense traffic is highly profitable, while a $40 trip covering 15 miles on a highway rapidly degrades hourly earnings and increases vehicle depreciation. Without spatial data intelligence, fleets cannot strategically position vehicles to maximize profit margins.

# Hypotheses
1. **The Spatial-Temporal Shift:** The most lucrative pickup zones are not static; they shift drastically from commercial hubs (Midtown Manhattan) during business hours to residential/leisure hubs (Brooklyn/West Village) during late-night hours.
2. **The "Airport Fallacy":** While airport pickups (JFK/LaGuardia) generate the highest total gross revenue per trip, their actual yield per mile is significantly lower than short, high-density intra-Manhattan trips.
3. **The Short-Trip Premium:** Trips under 3 miles generate a disproportionately higher yield per mile due to the heavy weighting of the initial base fare and congestion surcharges.

# Research Objectives
1. **Calculate Baseline Yield Metric:** To engineer a robust yield_per_mile financial metric that standardizes profitability across millions of trips of varying lengths and durations.

2. **Identify High-Yield Corridors:** To determine the top 10 specific neighborhoods (Location IDs) that consistently generate the highest yield per mile across the month.

3. **Map "Dead Zones":** To identify high-volume areas that deceptively generate low revenue-per-mile, highlighting zones drivers should actively avoid starting their shifts in.

4. **Temporal Fluctuation Analysis:** To track how pricing elasticity and yield degrade or spike depending on the hour of the day (Peak Rush Hour vs. Off-Peak).

5. **Inter-Borough vs. Intra-Borough Profitability:** To assess whether trips staying within the same borough (e.g., Manhattan to Manhattan) are mathematically more efficient than trips crossing bridges/boroughs.

6. **Fleet Positioning Strategy:** To synthesize these findings into a data-driven "shift deployment strategy" that advises fleet managers exactly where to deploy vehicles at 8:00 AM versus 6:00 PM for maximum ROI.

In [4]:
!pip install duckdb pandas

In [5]:
import duckdb
import pandas as pd

In [21]:
# Connect to the perfectly clean data we just built
con = duckdb.connect()
clean_data = "'data/cleaned_taxi_data_for_ml.parquet'"

In [41]:
# ---------------------------------------------------------
# OBJECTIVE 1 & 2: Top 10 High-Yield Corridors
# ---------------------------------------------------------

query_top_zones = f"""
with TripYields as (
select
     pickup_zone,
     pickup_borough,
     (total_amount/trip_distance) as yield_per_mile
from {clean_data}
where trip_distance > 0
)
select 
     pickup_zone,
     pickup_borough,
     count(*) as total_trips,
     round(avg(yield_per_mile),2) as avg_yield_per_mile
from TripYields
group by pickup_zone, pickup_borough
having count(*) > 1000
order by avg_yield_per_mile desc
limit 10;
"""

print("🏆 TOP 10 HIGH-YIELD CORRIDORS")
display(con.execute(query_top_zones).df())


🏆 TOP 10 HIGH-YIELD CORRIDORS


,pickup_zone,pickup_borough,total_trips,avg_yield_per_mile
0,South Ozone Park,Queens,1274,61.41
1,Jackson Heights,Queens,1383,60.90
2,Midtown Center,Manhattan,137422,16.69
3,Long Island City/Hunters Point,Queens,1854,16.53
4,Garment District,Manhattan,41486,16.40
5,Upper East Side South,Manhattan,152527,16.39
6,Midtown East,Manhattan,104971,16.34
7,Sutton Place/Turtle Bay North,Manhattan,49155,16.24
8,Midtown South,Manhattan,60094,16.23
9,Murray Hill,Manhattan,80486,15.95


In [45]:
# ---------------------------------------------------------
# OBJECTIVE 3: The "Dead Zones" (High Volume, Low Yield)
# ---------------------------------------------------------

query_dead_zones = f"""
with TripYields as (
select
     pickup_zone,
     pickup_borough,
     (total_amount/trip_distance) as yield_per_mile
from {clean_data}
where trip_distance > 0
)
select 
     pickup_zone,
     pickup_borough,
     count(*) as total_trips,
     round(avg(yield_per_mile),2) as avg_yield_per_mile
from TripYields
group by pickup_zone, pickup_borough
having count(*) > 5000
order by avg_yield_per_mile asc
limit 10;
"""

print("\n💀 TOP 10 DEAD ZONES (High Volume, Lowest Yield)")
display(con.execute(query_dead_zones).df())


💀 TOP 10 DEAD ZONES (High Volume, Lowest Yield)


,pickup_zone,pickup_borough,total_trips,avg_yield_per_mile
0,JFK Airport,Queens,144915,6.36
1,LaGuardia Airport,Queens,91776,8.07
2,East Elmhurst,Queens,7464,8.34
3,Central Harlem North,Manhattan,5607,8.73
4,East Harlem North,Manhattan,7226,9.77
5,Financial District South,Manhattan,8298,10.20
6,Central Harlem,Manhattan,8055,10.26
7,Morningside Heights,Manhattan,13242,10.90
8,Financial District North,Manhattan,13403,10.91
9,Battery Park City,Manhattan,15941,10.99


In [33]:
# ---------------------------------------------------------
# OBJECTIVE 4: Temporal Fluctuation (Time of Day Elasticity)
# ---------------------------------------------------------

query_temporal = f"""
with ShiftYields as (
select
     case
        when pickup_hour between 7 and 9 then '1_Morning_Rush (7a-10a)'
        when pickup_hour between 10 and 15 then '2_Midday (10a-4p)'
        when pickup_hour between 16 and 19 then '3_Evening_Rush (4p-8p)'
        else '4_Night/Overnight'
    end as business_shift,
    (total_amount/trip_distance) as yield_per_mile
from {clean_data}
)
select
     business_shift,
     count(*) as total_trips,
     round(avg(yield_per_mile),2) as avg_yield_per_mile
     from ShiftYields
     group by business_shift
     order by business_shift;
"""

print("\n⏰ TEMPORAL YIELD FLUCTUATION")
display(con.execute(query_temporal).df())


⏰ TEMPORAL YIELD FLUCTUATION


,business_shift,total_trips,avg_yield_per_mile
0,1_Morning_Rush (7a-10a),315705,13.19
1,2_Midday (10a-4p),961780,14.41
2,3_Evening_Rush (4p-8p),780776,15.60
3,4_Night/Overnight,826199,12.46


In [35]:
# ---------------------------------------------------------
# OBJECTIVE 5: Inter-Borough vs. Intra-Borough
# ---------------------------------------------------------
query_borough = f"""
    WITH BoroughFlags AS (
        SELECT 
            CASE 
                WHEN pickup_borough = dropoff_borough THEN 'Intra-Borough (Stayed in same borough)'
                ELSE 'Inter-Borough (Crossed bridges/boroughs)'
            END AS route_type,
            (total_amount / trip_distance) AS yield_per_mile
        FROM {clean_data}
    )
    SELECT 
        route_type,
        COUNT(*) AS total_trips,
        ROUND(AVG(yield_per_mile), 2) AS avg_yield_per_mile
    FROM BoroughFlags
    GROUP BY route_type
    ORDER BY avg_yield_per_mile DESC;
"""
print("\n🌉 INTER VS. INTRA-BOROUGH PROFITABILITY")
display(con.execute(query_borough).df())


🌉 INTER VS. INTRA-BOROUGH PROFITABILITY


,route_type,total_trips,avg_yield_per_mile
0,Intra-Borough (Stayed in same borough),2446000,15.39
1,Inter-Borough (Crossed bridges/boroughs),438460,6.47
